In [45]:
import polars as pl
import math
from pathlib import Path

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(250)

polars.config.Config

In [46]:
import polars as pl
from pathlib import Path
from datetime import timedelta

In [47]:
GTFS = Path("raw/processed_gtfs")
RT = Path("raw")
 
# ---------------------------------------------------
# Read files
# ---------------------------------------------------
 
routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet")
stop_times = pl.read_parquet(GTFS / "stop_times.parquet")
 
trip_updates = pl.read_parquet(RT / "trip_updates/2026-07-01.parquet")
vehicle_positions = pl.read_parquet(RT / "vehicle_positions/2026-07-01.parquet")
 
print("Loaded")

Loaded


In [48]:
def gtfs_to_seconds(col):
    p = pl.col(col).str.split(":")
    return (
        p.list.get(0).cast(pl.Int32) * 3600
        + p.list.get(1).cast(pl.Int32) * 60
        + p.list.get(2).cast(pl.Int32)
    )
 
stop_times = stop_times.with_columns(
    pl.col("stop_sequence").cast(pl.UInt32),
    pl.col("stop_id").cast(pl.Utf8),
    gtfs_to_seconds("arrival_time").alias("scheduled_arrival"),
    gtfs_to_seconds("departure_time").alias("scheduled_departure"),
)
 
stops = stops.with_columns([
    pl.col("stop_id").cast(pl.Utf8),

    # pl.col("stop_lat")
    #   .str.strip_chars()
    #   .cast(pl.Float64),

    # pl.col("stop_lon")
    #   .str.strip_chars()
    #   .cast(pl.Float64),
])

In [49]:

trip_updates = (
    trip_updates
    .sort("feed_timestamp")
    .group_by(["trip_id", "start_date", "stop_sequence"])
    .last()
)

In [50]:
if "schedule_relationship" in trip_updates.columns:
    trip_updates = trip_updates.filter(pl.col("schedule_relationship") == 0)
 

In [51]:
trip_updates = trip_updates.with_columns(
    pl.from_epoch("arrival_time", time_unit="s").alias("event_time")
)
 
vehicle_positions = vehicle_positions.with_columns(
    pl.from_epoch("timestamp", time_unit="s").alias("vehicle_time")
)

In [52]:
TRIP_ID_PATTERN = r'^[A-Z]{2}_([A-Z0-9]+)-(\w+?)-(\d+)_([A-Z0-9+]+)_(\d+)$'
 
def parse_trip_id(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 2).alias("_service_day"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 3).alias("_origin_secs"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 5).alias("_trip_num"),
    ])
 
trips_parsed = parse_trip_id(trips)
tu_parsed = parse_trip_id(trip_updates)
 
# 1) direct trip_id match
direct = tu_parsed.join(
    trips_parsed.select(["trip_id", "route_id", "direction_id", "shape_id", "service_id"]),
    on="trip_id", how="inner", suffix="_static",
)

In [53]:
unmatched = tu_parsed.join(direct.select("trip_id").unique(), on="trip_id", how="anti")
fallback = unmatched.join(
    trips_parsed.select([
        "trip_id", "route_id", "direction_id", "shape_id", "service_id",
        "_service_day", "_origin_secs", "_trip_num",
    ]).rename({"trip_id": "_static_trip_id"}),
    on=["route_id", "_service_day", "_origin_secs", "_trip_num"],
    how="inner",
    suffix="_static",
).with_columns(
    pl.col("_static_trip_id").alias("trip_id")   # use the STATIC trip_id downstream
).drop("_static_trip_id")
 
n_direct, n_fallback, n_total = direct.height, fallback.height, tu_parsed.height
print(f"[join] direct match: {n_direct:,} | fallback match: {n_fallback:,} | "
      f"total: {(n_direct + n_fallback) / n_total:.1%} of {n_total:,} rows")
 
direct = direct.drop(["_service_day", "_origin_secs", "_trip_num"])
fallback = fallback.drop(["_service_day", "_origin_secs", "_trip_num"])

[join] direct match: 70,053 | fallback match: 0 | total: 100.0% of 70,053 rows


In [54]:
direct = (
    direct.drop(["route_id", "direction_id"])
          .rename({"route_id_static": "route_id", "direction_id_static": "direction_id"})
)
fallback = fallback.drop("direction_id").rename({"direction_id_static": "direction_id"})
 
matched = pl.concat([direct, fallback.select(direct.columns)])

In [55]:
data = (
    matched
    .join(
        stop_times.select([
            "trip_id", "stop_sequence", "stop_id",
            "scheduled_arrival", "scheduled_departure",
        ]),
        on=["trip_id", "stop_sequence"],
        how="inner",   # inner now: every row here already has a resolved static trip_id,
                        # so a missing stop_times row means bad data, not an expected gap
    )
    .join(
        stops.select(["stop_id", "stop_lat", "stop_lon"]),
        on="stop_id",
        how="left",
    )
)
 
print(data.shape)

(70053, 22)


In [56]:
def haversine_expr(lat1, lon1, lat2, lon2) -> pl.Expr:
    R = 6371000.0
    lat1r, lat2r = lat1.radians(), lat2.radians()
    dlat = (lat2 - lat1).radians()
    dlon = (lon2 - lon1).radians()
    a = (dlat / 2).sin() ** 2 + lat1r.cos() * lat2r.cos() * (dlon / 2).sin() ** 2
    return 2 * R * a.sqrt().arcsin()
 
vehicle_positions = vehicle_positions.sort(["vehicle_id", "vehicle_time"])
vehicle_positions = vehicle_positions.with_columns([
    pl.col("latitude").shift(1).over("vehicle_id").alias("_prev_lat"),
    pl.col("longitude").shift(1).over("vehicle_id").alias("_prev_lon"),
    pl.col("vehicle_time").shift(1).over("vehicle_id").alias("_prev_time"),
])
vehicle_positions = vehicle_positions.with_columns(
    haversine_expr(pl.col("_prev_lat"), pl.col("_prev_lon"), pl.col("latitude"), pl.col("longitude")).alias("_dist_m")
)
vehicle_positions = vehicle_positions.with_columns(
    (
        pl.col("_dist_m")
        / (pl.col("vehicle_time") - pl.col("_prev_time")).dt.total_seconds().clip(lower_bound=1)
    ).alias("speed_mps")
)

In [57]:
vp = vehicle_positions.select([
    "vehicle_id", "vehicle_time", "latitude", "longitude", "bearing", "speed_mps",
])
 
data = data.sort(["vehicle_id", "event_time"])
vp = vp.sort(["vehicle_id", "vehicle_time"])
 
data = data.join_asof(
    vp,
    left_on="event_time",
    right_on="vehicle_time",
    by="vehicle_id",
    strategy="backward",
    tolerance=timedelta(minutes=2),
)

C:\Users\ishan\AppData\Local\Temp\ipykernel_28116\3507643840.py:8: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  data = data.join_asof(


In [58]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("arrival_time").shift(-1).over("trip_id").alias("next_arrival_time")
    )
    .with_columns(
        (pl.col("next_arrival_time") - pl.col("arrival_time")).alias("travel_time")
    )
)
 
data = data.filter(
    (pl.col("travel_time") > 0) & (pl.col("travel_time") < 1800)   # 30 min cap
)

In [59]:
# data = data.with_columns([
#     pl.col("event_time").dt.hour().alias("hour"),
#     pl.col("event_time").dt.weekday().alias("weekday"),
#     pl.col("event_time").dt.month().alias("month"),
# ])
 
# data = data.with_columns([
#     (
#         (pl.col("hour").is_between(7, 9)) |
#         (pl.col("hour").is_between(16, 18))
#     ).cast(pl.Int8).alias("is_peak")
# ])

data = data.with_columns(
    pl.col("event_time")
      .dt.replace_time_zone("UTC")
      .dt.convert_time_zone("America/New_York")
      .alias("event_time_local")
)
 
data = data.with_columns([
    pl.col("event_time_local").dt.hour().alias("hour"),
    pl.col("event_time_local").dt.weekday().alias("weekday"),
    pl.col("event_time_local").dt.month().alias("month"),
])
 
data = data.with_columns([
    (
        (pl.col("hour").is_between(7, 9)) |
        (pl.col("hour").is_between(16, 18))
    ).cast(pl.Int8).alias("is_peak")
])

In [60]:
data = data.with_columns([
    (
        pl.col("scheduled_arrival")
        - pl.col("scheduled_departure").shift(1).over("trip_id")
    ).alias("scheduled_segment_time"),
 
    (
        pl.col("stop_sequence") / pl.col("stop_sequence").max().over("trip_id")
    ).alias("trip_progress"),
])

In [61]:
segment_network = pl.read_parquet("processed/segment_network.parquet")
 
# dtype fix: segment_network's stop_id/next_stop_id are Int64 (straight
# from stop_times.txt), but `data`'s stop_id was cast to Utf8 earlier for
# the RT join - cast both sides to match before joining.
segment_network = segment_network.with_columns([
    pl.col("stop_id").cast(pl.Utf8),
    pl.col("next_stop_id").cast(pl.Utf8),
])

In [62]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("stop_id").shift(-1).over("trip_id").alias("next_stop_id")
    )
)
 
# %%
n_before = data.height
 
data = data.join(
    segment_network.select([
        "shape_id", "stop_id", "next_stop_id",
        "segment_length", "scheduled_travel_time",
    ]).rename({"scheduled_travel_time": "segment_scheduled_travel_time"}),
    on=["shape_id", "stop_id", "next_stop_id"],
    how="left",
)
 
print(f"rows before: {n_before:,} | after segment join: {data.height:,}")
print("null segment_length rows:", data.filter(pl.col("segment_length").is_null()).height)
 

rows before: 67,196 | after segment join: 67,196
null segment_length rows: 2949


In [63]:
data = data.with_columns(
    (pl.col("segment_length") / pl.col("segment_scheduled_travel_time").clip(lower_bound=1))
    .alias("scheduled_segment_speed_mps")
)

In [64]:
data = data.sort(["trip_id", "stop_sequence"])
check = data.select([
    "trip_id", "stop_sequence", "scheduled_segment_time", "segment_scheduled_travel_time",
]).with_columns(
    # bring segment_scheduled_travel_time from row k up to align with
    # scheduled_segment_time at row k+1 (both now describe segment k->k+1)
    pl.col("segment_scheduled_travel_time").shift(1).over("trip_id").alias("segment_scheduled_travel_time_aligned")
).drop_nulls(subset=["scheduled_segment_time", "segment_scheduled_travel_time_aligned"])
 
diff = (check["scheduled_segment_time"] - check["segment_scheduled_travel_time_aligned"]).abs()
print("median abs diff (aligned):", diff.median())
print("rows with diff > 30s (aligned):", (diff > 30).sum(), "/", check.height)
 
# expected: last stop of each trip has null segment_length/next_stop_id -
# should be roughly one per trip, not a bug
n_null_segment = data.filter(pl.col("segment_length").is_null()).height
n_trips = data["trip_id"].n_unique()
print(f"null segment_length rows: {n_null_segment:,} vs trip count: {n_trips:,} (should be close)")
 

median abs diff (aligned): 20.0
rows with diff > 30s (aligned): 21580 / 64247
null segment_length rows: 2,949 vs trip count: 1,314 (should be close)


In [65]:

import requests
from datetime import datetime
 
STATION_LAT, STATION_LON = 40.7829, -73.9654  # Central Park
 
dates_needed = data["start_date"].unique().sort().to_list()
start_date = str(dates_needed[0])
end_date = str(dates_needed[-1])
start_fmt = f"{start_date[:4]}-{start_date[4:6]}-{start_date[6:8]}"
end_fmt = f"{end_date[:4]}-{end_date[4:6]}-{end_date[6:8]}"
 
print(f"Fetching weather for {start_fmt} to {end_fmt}")
 
resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": STATION_LAT,
        "longitude": STATION_LON,
        "start_date": start_fmt,
        "end_date": end_fmt,
        "hourly": "temperature_2m,precipitation,rain,snowfall,"
                  "windspeed_10m,weathercode",
        "timezone": "America/New_York",
    },
    timeout=30,
)
resp.raise_for_status()
weather_json = resp.json()["hourly"]
 
weather = pl.DataFrame({
    "weather_time_str": weather_json["time"],
    "temperature_c": weather_json["temperature_2m"],
    "precipitation_mm": weather_json["precipitation"],
    "rain_mm": weather_json["rain"],
    "snowfall_cm": weather_json["snowfall"],
    "windspeed_kmh": weather_json["windspeed_10m"],
    "weathercode": weather_json["weathercode"],
})
 
weather = weather.with_columns(
    pl.col("weather_time_str").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M").alias("weather_time")
).with_columns([
    pl.col("weather_time").dt.strftime("%Y%m%d").alias("start_date"),  # keep as str - matches data's dtype
    pl.col("weather_time").dt.hour().alias("hour"),
])
 
print(weather.shape)
print(weather.head())

Fetching weather for 2026-06-30 to 2026-07-01
(48, 10)
shape: (5, 10)
┌──────────────────┬───────────────┬──────────────────┬─────────┬─────────────┬───────────────┬─────────────┬─────────────────────┬────────────┬──────┐
│ weather_time_str ┆ temperature_c ┆ precipitation_mm ┆ rain_mm ┆ snowfall_cm ┆ windspeed_kmh ┆ weathercode ┆ weather_time        ┆ start_date ┆ hour │
│ ---              ┆ ---           ┆ ---              ┆ ---     ┆ ---         ┆ ---           ┆ ---         ┆ ---                 ┆ ---        ┆ ---  │
│ str              ┆ f64           ┆ f64              ┆ f64     ┆ f64         ┆ f64           ┆ i64         ┆ datetime[μs]        ┆ str        ┆ i8   │
╞══════════════════╪═══════════════╪══════════════════╪═════════╪═════════════╪═══════════════╪═════════════╪═════════════════════╪════════════╪══════╡
│ 2026-06-30T00:00 ┆ 24.6          ┆ 0.0              ┆ 0.0     ┆ 0.0         ┆ 5.7           ┆ 1           ┆ 2026-06-30 00:00:00 ┆ 20260630   ┆ 0    │
│ 2026-06-30T01:00

In [66]:
# %%
# ---------------------------------------------------------------------
# Join weather onto data by (start_date, hour) - both now NYC-local
# ---------------------------------------------------------------------
 
n_before = data.height
 
data = data.join(
    weather.select([
        "start_date", "hour", "temperature_c", "precipitation_mm",
        "rain_mm", "snowfall_cm", "windspeed_kmh", "weathercode",
    ]),
    on=["start_date", "hour"],
    how="left",
)
 
print(f"rows before: {n_before:,} | after weather join: {data.height:,}")
print("null weather rows:", data.filter(pl.col("temperature_c").is_null()).height)

rows before: 67,196 | after weather join: 67,196
null weather rows: 0


In [67]:
data = data.with_columns([
    (pl.col("precipitation_mm") > 0.1).cast(pl.Int8).alias("is_raining"),
    (pl.col("snowfall_cm") > 0.0).cast(pl.Int8).alias("is_snowing"),
    # WMO weather codes 45 (fog) / 48 (depositing rime fog) as a
    # low-visibility proxy - the archive API doesn't expose visibility
    # directly (ERA5 reanalysis has no such variable; that's a
    # forecast-API-only field), so this is the closest available signal.
    pl.col("weathercode").is_in([45, 48]).cast(pl.Int8).alias("is_fog"),
])

In [68]:
# 1. UPSTREAM DELAY
# ---------------------------------------------------------------------
# scheduled_arrival is seconds-since-midnight (local, naive), while
# arrival_time (RT) is an absolute UTC epoch - not directly comparable
# yet. Convert scheduled_arrival to an absolute epoch first, anchored on
# start_date in America/New_York (same pattern used for the weather join).
 
data = data.with_columns(
    pl.col("start_date").str.strptime(pl.Date, "%Y%m%d").alias("service_date")
)
 
data = data.with_columns(
    (
        pl.col("service_date").cast(pl.Datetime)
          .dt.replace_time_zone("America/New_York")
        + pl.duration(seconds=pl.col("scheduled_arrival"))
    ).dt.epoch(time_unit="s").alias("scheduled_arrival_epoch")
)
 
data = data.with_columns(
    (pl.col("arrival_time") - pl.col("scheduled_arrival_epoch")).alias("delay_seconds")
)

In [69]:
# ---------------------------------------------------------------------
# Load + reshape ridership CSV - route + hour granularity
# ---------------------------------------------------------------------
ridership_raw = pl.read_csv(
    "MTA_Bus_Hourly_Ridership_Mar_July.csv",
    schema_overrides={"ridership": pl.Utf8, "transfers": pl.Utf8},
)

# ridership/transfers arrive as strings with thousands separators
# (e.g. "1,131") - strip commas before casting to Int64
ridership_raw = ridership_raw.with_columns(
    pl.col("transit_timestamp")
      .str.strptime(pl.Datetime, "%m/%d/%Y %I:%M:%S %p")
      .alias("_ts"),
    pl.col("ridership").str.replace_all(",", "").cast(pl.Int64),
    pl.col("transfers").str.replace_all(",", "").cast(pl.Int64),
).with_columns([
    pl.col("_ts").dt.date().alias("service_date"),
    pl.col("_ts").dt.hour().alias("hour"),
])

# collapse both fare_class_category and payment_method - one total per
# (route, date, hour)
ridership_hourly = (
    ridership_raw
    .group_by(["bus_route", "service_date", "hour"])
    .agg([
        pl.col("ridership").sum(),
        pl.col("transfers").sum(),
    ])
)

print(ridership_hourly.shape)
print(ridership_hourly.head(10))


(16320, 5)
shape: (10, 5)
┌───────────┬──────────────┬──────┬───────────┬───────────┐
│ bus_route ┆ service_date ┆ hour ┆ ridership ┆ transfers │
│ ---       ┆ ---          ┆ ---  ┆ ---       ┆ ---       │
│ str       ┆ date         ┆ i8   ┆ i64       ┆ i64       │
╞═══════════╪══════════════╪══════╪═══════════╪═══════════╡
│ M15       ┆ 2026-03-20   ┆ 4    ┆ 100       ┆ 21        │
│ M1        ┆ 2026-06-27   ┆ 22   ┆ 82        ┆ 13        │
│ M2        ┆ 2026-05-10   ┆ 21   ┆ 98        ┆ 10        │
│ M1        ┆ 2026-07-07   ┆ 9    ┆ 488       ┆ 129       │
│ M4        ┆ 2026-04-10   ┆ 8    ┆ 775       ┆ 195       │
│ M101      ┆ 2026-05-09   ┆ 16   ┆ 498       ┆ 116       │
│ M101      ┆ 2026-06-01   ┆ 11   ┆ 770       ┆ 209       │
│ M15       ┆ 2026-07-09   ┆ 22   ┆ 363       ┆ 68        │
│ M1        ┆ 2026-03-11   ┆ 22   ┆ 68        ┆ 7         │
│ M4        ┆ 2026-04-08   ┆ 11   ┆ 521       ┆ 111       │
└───────────┴──────────────┴──────┴───────────┴───────────┘


In [70]:
# ---------------------------------------------------------------------
# Join onto `data` by (route_id, service_date, hour).
# NOTE: this ridership file only covers 5 routes (M1, M2, M4, M15, M101).
# route_id naming is verified before joining - if your GTFS route_id has
# an agency prefix (e.g. "MTA NYCT_M1") rather than the bare "M1" used in
# the ridership file, normalize one side before joining.
# ---------------------------------------------------------------------
sample_route_ids = data.select("route_id").unique().sort("route_id").head(10)
print("sample route_id values in `data`:", sample_route_ids["route_id"].to_list())
print("route values in ridership file:", ridership_hourly["bus_route"].unique().sort().to_list())

# If the printed lists above don't share the same format (e.g. one has a
# prefix like "MTA NYCT_"), normalize here before the join, e.g.:
# data = data.with_columns(pl.col("route_id").str.extract(r"([A-Z]+\d+)$", 1).alias("route_id"))

n_before = data.height

data = data.join(
    ridership_hourly,
    left_on=["route_id", "service_date", "hour"],
    right_on=["bus_route", "service_date", "hour"],
    how="left",
).with_columns([
    pl.col("ridership").fill_null(0),
    pl.col("transfers").fill_null(0),
])

print(f"rows before: {n_before:,} | after ridership join: {data.height:,}")
print("routes with zero-filled ridership (not in ridership file, or no match):",
      data.filter(pl.col("ridership") == 0).select("route_id").unique().height)


sample route_id values in `data`: ['M1', 'M101', 'M15', 'M2', 'M4']
route values in ridership file: ['M1', 'M101', 'M15', 'M2', 'M4']
rows before: 67,196 | after ridership join: 67,196
routes with zero-filled ridership (not in ridership file, or no match): 0


In [71]:
print(data.select(
    pl.col("delay_seconds").min().alias("min"),
    pl.col("delay_seconds").max().alias("max"),
    pl.col("delay_seconds").median().alias("median"),
))

shape: (1, 3)
┌───────┬───────┬────────┐
│ min   ┆ max   ┆ median │
│ ---   ┆ ---   ┆ ---    │
│ i64   ┆ i64   ┆ f64    │
╞═══════╪═══════╪════════╡
│ -1263 ┆ 13450 ┆ 67.0   │
└───────┴───────┴────────┘


In [72]:
data = data.sort(["trip_id", "stop_sequence"])
data = data.with_columns(
    pl.col("delay_seconds").shift(1).over("trip_id").alias("upstream_delay_seconds")
)


In [73]:
# %%
# ---------------------------------------------------------------------
# 3. HEADWAY TO PRECEDING BUS (same route + direction + stop)
# ---------------------------------------------------------------------
# For each (route_id, direction_id, stop_id), sort arrivals by time and
# take the gap to the PREVIOUS bus that hit the same stop - this is a
# same-route self-join across TRIPS, not within a single trip.
 
headway_base = (
    data.select(["route_id", "direction_id", "stop_id", "trip_id", "arrival_time"])
    .unique()
    .sort(["route_id", "direction_id", "stop_id", "arrival_time"])
)
 
headway_base = headway_base.with_columns(
    pl.col("arrival_time")
      .shift(1)
      .over(["route_id", "direction_id", "stop_id"])
      .alias("_prev_bus_arrival")
)
headway_base = headway_base.with_columns(
    (pl.col("arrival_time") - pl.col("_prev_bus_arrival")).alias("headway_seconds")
)

In [74]:
data = data.join(
    headway_base.select(["route_id", "direction_id", "stop_id", "trip_id", "arrival_time", "headway_seconds"]),
    on=["route_id", "direction_id", "stop_id", "trip_id", "arrival_time"],
    how="left",
)

In [75]:
print(data.select(
    pl.col("headway_seconds").min().alias("min"),
    pl.col("headway_seconds").max().alias("max"),
    pl.col("headway_seconds").median().alias("median"),
))
print("null headway rows (first bus of the day at that stop):",
      data.filter(pl.col("headway_seconds").is_null()).height)
 

shape: (1, 3)
┌─────┬───────┬────────┐
│ min ┆ max   ┆ median │
│ --- ┆ ---   ┆ ---    │
│ i64 ┆ i64   ┆ f64    │
╞═════╪═══════╪════════╡
│ 0   ┆ 49277 ┆ 638.0  │
└─────┴───────┴────────┘
null headway rows (first bus of the day at that stop): 692


In [76]:
calendar_df = pl.read_parquet("calendar_dataset.parquet")
calendar_df = calendar_df.with_columns(
    pl.col("service_date").str.strptime(pl.Date, "%Y-%m-%d")
)

calendar_df = calendar_df.select([
    "service_date",
    "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "major_event_count",
]).with_columns(
    (pl.col("major_event_count") > 0).alias("has_major_event")
)
 
n_before = data.height
 
data = data.join(calendar_df, on="service_date", how="left")
 
print(f"rows before: {n_before:,} | after calendar join: {data.height:,}")
print("null calendar rows (service_date outside 2026-03-01 to 2026-07-17):",
      data.filter(pl.col("is_federal_holiday").is_null()).height)

rows before: 67,196 | after calendar join: 67,196
null calendar rows (service_date outside 2026-03-01 to 2026-07-17): 0


In [77]:
data

trip_id,start_date,stop_sequence,feed_timestamp,fetch_timestamp,start_time,schedule_relationship,vehicle_id,trip_timestamp,stop_id,arrival_time,departure_time,event_time,route_id,direction_id,shape_id,service_id,stop_id_right,scheduled_arrival,scheduled_departure,stop_lat,stop_lon,vehicle_time,latitude,longitude,bearing,speed_mps,next_arrival_time,travel_time,event_time_local,hour,weekday,month,is_peak,scheduled_segment_time,trip_progress,next_stop_id,segment_length,segment_scheduled_travel_time,scheduled_segment_speed_mps,temperature_c,precipitation_mm,rain_mm,snowfall_cm,windspeed_kmh,weathercode,is_raining,is_snowing,is_fog,service_date,scheduled_arrival_epoch,delay_seconds,ridership,transfers,upstream_delay_seconds,headway_seconds,is_weekend,is_federal_holiday,is_school_day,major_event_count,has_major_event
str,str,u32,u64,"datetime[μs, UTC]",str,i32,str,u64,str,i64,i64,datetime[μs],str,i64,str,str,str,i32,i32,f64,f64,datetime[μs],f32,f32,f32,f64,i64,i64,"datetime[μs, America/New_York]",i8,i8,i8,i8,i32,f64,str,f64,i64,f64,f64,f64,f64,f64,f64,i64,i8,i8,i8,date,i64,i64,i64,i64,i64,i64,bool,bool,bool,u32,bool
"""MV_C6-Weekday-110900_M4_447""","""20260630""",70,1782864003,2026-07-01 00:00:28.366841 UTC,"""""",0,"""MTA NYCT_9760""",1782863994,"""400629""",1782864018,1782864018,2026-07-01 00:00:18,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400629""",71825,71825,40.847906,-73.939407,2026-06-30 23:59:54,40.847366,-73.939713,67.574135,null,1782864104,86,2026-06-30 20:00:18 EDT,20,2,6,0,null,0.921053,"""400630""",153.41805,35,4.383373,31.1,0.0,0.0,0.0,9.5,2,0,0,0,2026-06-30,1782863825,193,267,45,null,null,false,false,false,0,false
"""MV_C6-Weekday-110900_M4_447""","""20260630""",71,1782864107,2026-07-01 00:01:58.841053 UTC,"""""",0,"""MTA NYCT_9760""",1782864084,"""400630""",1782864104,1782864104,2026-07-01 00:01:44,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400630""",71869,71869,40.849229,-73.938932,2026-07-01 00:01:24,40.849068,-73.939072,61.927513,0.578304,1782864181,77,2026-06-30 20:01:44 EDT,20,2,6,0,44,0.934211,"""400631""",230.315978,52,4.429153,31.1,0.0,0.0,0.0,9.5,2,0,0,0,2026-06-30,1782863869,235,267,45,193,null,false,false,false,0,false
"""MV_C6-Weekday-110900_M4_447""","""20260630""",72,1782864180,2026-07-01 00:03:28.367952 UTC,"""""",0,"""MTA NYCT_9760""",1782864174,"""400631""",1782864181,1782864181,2026-07-01 00:03:01,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400631""",71934,71934,40.851202,-73.938078,2026-07-01 00:02:54,40.851707,-73.937981,69.819817,0.470303,1782864238,57,2026-06-30 20:03:01 EDT,20,2,6,0,65,0.947368,"""400632""",137.22478,31,4.426606,31.1,0.0,0.0,0.0,9.5,2,0,0,0,2026-06-30,1782863934,247,267,45,235,null,false,false,false,0,false
"""MV_C6-Weekday-110900_M4_447""","""20260630""",73,1782864253,2026-07-01 00:04:28.367085 UTC,"""""",0,"""MTA NYCT_9760""",1782864234,"""400632""",1782864238,1782864238,2026-07-01 00:03:58,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400632""",71973,71973,40.852396,-73.937654,2026-07-01 00:03:54,40.852718,-73.93763,72.790443,1.528495,1782864275,37,2026-06-30 20:03:58 EDT,20,2,6,0,39,0.960526,"""400633""",141.957749,32,4.43618,31.1,0.0,0.0,0.0,9.5,2,0,0,0,2026-06-30,1782863973,265,267,45,247,null,false,false,false,0,false
"""MV_C6-Weekday-110900_M4_447""","""20260630""",74,1782864285,2026-07-01 00:04:58.686397 UTC,"""""",0,"""MTA NYCT_9760""",1782864264,"""400633""",1782864275,1782864275,2026-07-01 00:04:35,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400633""",72013,72013,40.853639,-73.937266,2026-07-01 00:04:24,40.85331,-73.937447,72.790443,2.25089,1782864328,53,2026-06-30 20:04:35 EDT,20,2,6,0,40,0.973684,"""400634""",165.220801,38,4.347916,31.1,0.0,0.0,0.0,9.5,2,0,0,0,2026-06-30,1782864013,262,267,45,265,null,false,false,false,0,false
"""MV_C6-Weekday-110900_M4_447""","""20260630""",75,1782864337,2026-07-01 00:05:58.545280 UTC,"""""",0,"""MTA NYCT_9760""",1782864324,"""400634""",1782864328,1782864328,2026-07-01 00:05:28,"""M4""",0,"""M040923""","""MV

In [78]:
data.shape

(67196, 61)

In [79]:
data.null_count()

trip_id,start_date,stop_sequence,feed_timestamp,fetch_timestamp,start_time,schedule_relationship,vehicle_id,trip_timestamp,stop_id,arrival_time,departure_time,event_time,route_id,direction_id,shape_id,service_id,stop_id_right,scheduled_arrival,scheduled_departure,stop_lat,stop_lon,vehicle_time,latitude,longitude,bearing,speed_mps,next_arrival_time,travel_time,event_time_local,hour,weekday,month,is_peak,scheduled_segment_time,trip_progress,next_stop_id,segment_length,segment_scheduled_travel_time,scheduled_segment_speed_mps,temperature_c,precipitation_mm,rain_mm,snowfall_cm,windspeed_kmh,weathercode,is_raining,is_snowing,is_fog,service_date,scheduled_arrival_epoch,delay_seconds,ridership,transfers,upstream_delay_seconds,headway_seconds,is_weekend,is_federal_holiday,is_school_day,major_event_count,has_major_event
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,6230,6230,6230,6230,6284,0,0,0,0,0,0,0,1314,0,1314,2949,2949,2949,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1314,692,0,0,0,0,0


In [80]:
features = [
    "route_id",
    "direction_id",
    "shape_id",
    "service_id",
 
    "stop_sequence",
    "trip_progress",
 
    "hour",
    "weekday",
    "month",
    "is_peak",
 
    "scheduled_arrival",
    "scheduled_departure",
    "scheduled_segment_time",
 
    "stop_lat",
    "stop_lon",
 
    "latitude",
    "longitude",
    "bearing",
    "temperature_c",
    "precipitation_mm",
    "snowfall_cm",
    "windspeed_kmh",
    "is_raining",
    "is_snowing",
    "is_fog",
    "weathercode",
    "segment_length",
    "scheduled_segment_speed_mps",
    "upstream_delay_seconds",
    "speed_mps",
    "headway_seconds",
    "is_weekend",
    "is_federal_holiday",
    "is_school_day",
    "has_major_event",

    
    "ridership",
    "transfers",
]
target = "travel_time"
id_cols = ["trip_id", "start_date"]
 
print(data.columns)
 
# %%
model_ready = data.select(id_cols + features + [target]).drop_nulls(subset=features + [target])
 
print(model_ready.shape)
print(model_ready.null_count())
print(model_ready.head())

['trip_id', 'start_date', 'stop_sequence', 'feed_timestamp', 'fetch_timestamp', 'start_time', 'schedule_relationship', 'vehicle_id', 'trip_timestamp', 'stop_id', 'arrival_time', 'departure_time', 'event_time', 'route_id', 'direction_id', 'shape_id', 'service_id', 'stop_id_right', 'scheduled_arrival', 'scheduled_departure', 'stop_lat', 'stop_lon', 'vehicle_time', 'latitude', 'longitude', 'bearing', 'speed_mps', 'next_arrival_time', 'travel_time', 'event_time_local', 'hour', 'weekday', 'month', 'is_peak', 'scheduled_segment_time', 'trip_progress', 'next_stop_id', 'segment_length', 'segment_scheduled_travel_time', 'scheduled_segment_speed_mps', 'temperature_c', 'precipitation_mm', 'rain_mm', 'snowfall_cm', 'windspeed_kmh', 'weathercode', 'is_raining', 'is_snowing', 'is_fog', 'service_date', 'scheduled_arrival_epoch', 'delay_seconds', 'ridership', 'transfers', 'upstream_delay_seconds', 'headway_seconds', 'is_weekend', 'is_federal_holiday', 'is_school_day', 'major_event_count', 'has_major_e

In [81]:
model_ready.columns

['trip_id',
 'start_date',
 'route_id',
 'direction_id',
 'shape_id',
 'service_id',
 'stop_sequence',
 'trip_progress',
 'hour',
 'weekday',
 'month',
 'is_peak',
 'scheduled_arrival',
 'scheduled_departure',
 'scheduled_segment_time',
 'stop_lat',
 'stop_lon',
 'latitude',
 'longitude',
 'bearing',
 'temperature_c',
 'precipitation_mm',
 'snowfall_cm',
 'windspeed_kmh',
 'is_raining',
 'is_snowing',
 'is_fog',
 'weathercode',
 'segment_length',
 'scheduled_segment_speed_mps',
 'upstream_delay_seconds',
 'speed_mps',
 'headway_seconds',
 'is_weekend',
 'is_federal_holiday',
 'is_school_day',
 'has_major_event',
 'ridership',
 'transfers',
 'travel_time']

In [82]:
model_ready

trip_id,start_date,route_id,direction_id,shape_id,service_id,stop_sequence,trip_progress,hour,weekday,month,is_peak,scheduled_arrival,scheduled_departure,scheduled_segment_time,stop_lat,stop_lon,latitude,longitude,bearing,temperature_c,precipitation_mm,snowfall_cm,windspeed_kmh,is_raining,is_snowing,is_fog,weathercode,segment_length,scheduled_segment_speed_mps,upstream_delay_seconds,speed_mps,headway_seconds,is_weekend,is_federal_holiday,is_school_day,has_major_event,ridership,transfers,travel_time
str,str,str,i64,str,str,u32,f64,i8,i8,i8,i8,i32,i32,i32,f64,f64,f32,f32,f32,f64,f64,f64,f64,i8,i8,i8,i64,f64,f64,i64,f64,i64,bool,bool,bool,bool,i64,i64,i64
"""MV_C6-Weekday-111700_M3_326""","""20260630""","""M2""",0,"""M020092""","""MV_C6-Weekday""",41,0.911111,20,2,6,0,71622,71622,70,40.832912,-73.939461,40.833019,-73.939445,47.511364,31.1,0.0,0.0,9.5,0,0,0,2,168.383636,3.061521,608,72.608229,8,false,false,false,false,226,35,40
"""MV_C6-Weekday-111700_M3_326""","""20260630""","""M2""",0,"""M020092""","""MV_C6-Weekday""",42,0.933333,20,2,6,0,71670,71670,48,40.834045,-73.938133,40.835564,-73.937431,65.207558,31.1,0.0,0.0,9.5,0,0,0,2,142.313135,3.027939,586,8.592368,9,false,false,false,false,226,35,5
"""MV_C6-Weekday-111700_M3_326""","""20260630""","""M2""",0,"""M020092""","""MV_C6-Weekday""",43,0.955556,20,2,6,0,71711,71711,41,40.835242,-73.937496,40.835564,-73.937431,65.207558,31.1,0.0,0.0,9.5,0,0,0,2,166.574166,3.028621,578,8.592368,5,false,false,false,false,226,35,56
"""MV_C6-Weekday-111700_M3_326""","""20260630""","""M2""",0,"""M020092""","""MV_C6-Weekday""",44,0.977778,20,2,6,0,71758,71758,47,40.836657,-73.936852,40.836197,-73.937149,65.556046,31.1,0.0,0.0,9.5,0,0,0,2,204.714537,3.010508,542,2.477002,8,false,false,false,false,226,35,90
"""MV_C6-Weekday-111700_M5_528""","""20260630""","""M4""",0,"""M040923""","""MV_C6-Weekday""",70,0.921053,20,2,6,0,72305,72305,35,40.847906,-73.939407,40.848469,-73.939308,72.299576,31.1,0.0,0.0,9.5,0,0,0,2,153.41805,4.383373,-196,8.364308,100,false,false,false,false,267,45,130
"""MV_C6-Weekday-111700_M5_528""","""20260630""","""M4""",0,"""M040923""","""MV_C6-Weekday""",71,0.934211,20,2,6,0,72349,72349,44,40.849229,-73.938932,40.849243,-73.938972,61.488754,31.1,0.0,0.0,9.5,0,0,0,2,230.315978,4.429153,-187,74.105947,144,false,false,false,false,267,45,53
"""MV_C6-Weekday-111700_M5_528""","""20260630""","""M4""",0,"""M040923""","""MV_C6-Weekday""",72,0.947368,20,2,6,0,72414,72414,65,40.851202,-73.938078,40.851097,-73.93821,69.483452,31.1,0.0,0.0,9.5,0,0,0,2,137.22478,4.426606,-101,7.445031,120,false,false,false,false,267,45,40
"""MV_C6-Weekday-111700_M5_528""","""20260630""","""M4""",0,"""M040923""","""MV_C6-Weekday""",73,0.960526,20,2,6,0,72453,72453,39,40.852396,-73.937654,40.85247,-73.937706,72.790443,31.1,0.0,0.0,9.5,0,0,0,2,141.957749,4.43618,-113,2.881224,103,false,false,false,false,267,45,32
"""MV_C6-Weekday-111700_M5_528""","""20260630""","""M4""",0,"""M040923""","""MV_C6-Weekday""",74,0.973684,20,2,6,0,72493,72493,40,40.853639,-73.937266,40.853245,-73.937462,72.790443,31.1,0.0,0.0,9.5,0,0,0,2,165.220801,4.347916,-112,3.1615,98,false,false,false,false,267,45,42


In [83]:
print(model_ready.describe())

shape: (9, 41)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┐
│ sta ┆ tri ┆ sta ┆ rou ┆ dir ┆ sha ┆ ser ┆ sto ┆ tri ┆ hou ┆ wee ┆ mon ┆ is_ ┆ sch ┆ sch ┆ sch ┆ sto ┆ sto ┆ lat ┆ lon ┆ bea ┆ tem ┆ pre ┆ sno ┆ win ┆ is_ ┆ is_ ┆ is_ ┆ wea ┆ seg ┆ sch ┆ ups ┆ spe ┆ hea ┆ is_ ┆ is_ ┆ is_ ┆ has ┆ rid ┆ tra ┆ travel │
│ tis ┆ p_i ┆ rt_ ┆ te_ ┆ ect ┆ pe_ ┆ vic ┆ p_s ┆ p_p ┆ r   ┆ kda ┆ th  ┆ pea ┆ edu ┆ edu ┆ edu ┆ p_l ┆ p_l ┆ itu ┆ git ┆ rin ┆ per ┆ cip ┆ wfa ┆ dsp ┆ rai ┆ sno ┆ fog ┆ the ┆ men ┆ edu ┆ tre ┆ ed_ ┆ dwa ┆ wee ┆ fed ┆ sch ┆ _ma ┆ ers ┆ nsf ┆ _time  │
│ tic ┆ d   ┆ dat ┆ id  ┆ ion ┆ id  ┆ e_i ┆ equ ┆ rog ┆ --- ┆ y   ┆ --- ┆ k   ┆ led ┆ led ┆ led ┆ at  ┆ on  ┆ de  ┆ ude ┆ g   ┆ atu ┆ ita ┆ ll_ ┆ eed ┆ nin ┆ win ┆ --- ┆ rco ┆ t_l ┆ led ┆ am_ ┆ mps ┆ y_s ┆ ken ┆ era ┆ ool ┆ jor ┆ hi

In [84]:
model_ready.group_by("route_id").len().sort("len", descending=True)

route_id,len
str,u32
"""M4""",14015
"""M101""",12422
"""M15""",11066
"""M1""",10671
"""M2""",8590


In [85]:
model_ready.select([
    pl.col("scheduled_arrival").min().alias("min"),
    pl.col("scheduled_arrival").max().alias("max"),
    pl.col("scheduled_arrival").mean().alias("mean"),
])

min,max,mean
i32,i32,f64
1241,95134,51595.764358


In [86]:
print(
    trip_updates.group_by("route_id").len().sort("len", descending=True)
)

print(
    matched.group_by("route_id").len().sort("len", descending=True)
)

print(
    model_ready.group_by("route_id").len().sort("len", descending=True)
)

shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 16698 │
│ M101     ┆ 15665 │
│ M15      ┆ 14044 │
│ M1       ┆ 13106 │
│ M2       ┆ 10540 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 16698 │
│ M101     ┆ 15665 │
│ M15      ┆ 14044 │
│ M1       ┆ 13106 │
│ M2       ┆ 10540 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 14015 │
│ M101     ┆ 12422 │
│ M15      ┆ 11066 │
│ M1       ┆ 10671 │
│ M2       ┆ 8590  │
└──────────┴───────┘


In [87]:
model_ready.write_parquet("raw/processed_gtfs/baseline_dataset.parquet")
print("written")

written
